# House Price Prediction Using Linear Regression

**Task 1 — Internship Project**

Goal: estimate a house's sale price from above-ground living area, bedroom count, and bathroom equivalents using Kaggle's *House Prices - Advanced Regression Techniques* training data.

`train.csv`, `test.csv`, `sample_submission.csv`, and `data_description.txt` are included in the project `data/` folder. The notebook reads `data/train.csv`. Do not use `test.csv` for training or evaluation because it has no `SalePrice` target column.


## 1. Import libraries

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path('data/train.csv')
OUTPUT_DIR = Path('artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Libraries imported.')

## 2. Load and inspect the dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError('Expected the included Kaggle dataset at data/train.csv')

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())
required_columns = ['GrLivArea', 'BedroomAbvGr', 'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath', 'SalePrice']
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
display(df[required_columns].describe())


## 3. Explore the target and feature relationships

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df['SalePrice'].dropna(), bins=30, edgecolor='black')
ax.set_title('Distribution of House Sale Prices')
ax.set_xlabel('Sale price (USD)')
ax.set_ylabel('Number of houses')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df['GrLivArea'], df['SalePrice'], alpha=0.6)
ax.set_title('Living Area vs. Sale Price')
ax.set_xlabel('Above-ground living area (sq ft)')
ax.set_ylabel('Sale price (USD)')
plt.tight_layout()
plt.show()

## 4. Prepare the model features

We use three predictors. The derived `TotalBaths` feature is:

`TotalBaths = FullBath + 0.5 × HalfBath + BsmtFullBath + 0.5 × BsmtHalfBath`

Each half-bath counts as 0.5 of a full bathroom, and basement bathrooms are included. Missing bathroom component values are treated as zero; other missing feature values are imputed with the median by the model pipeline.

In [ ]:
data = df.dropna(subset=['SalePrice']).copy()
X = pd.DataFrame(index=data.index)
X['GrLivArea'] = pd.to_numeric(data['GrLivArea'], errors='coerce')
X['BedroomAbvGr'] = pd.to_numeric(data['BedroomAbvGr'], errors='coerce')
baths = data[['FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath']].apply(pd.to_numeric, errors='coerce').fillna(0)
X['TotalBaths'] = baths['FullBath'] + 0.5*baths['HalfBath'] + baths['BsmtFullBath'] + 0.5*baths['BsmtHalfBath']
y = pd.to_numeric(data['SalePrice'], errors='coerce')
valid = y.notna()
X, y = X.loc[valid], y.loc[valid]
print('Rows used:', len(X))
print('Features:', list(X.columns))
display(X.head())

## 5. Split the data and train the model

The data is split into 80% training and 20% testing. Setting `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('regressor', LinearRegression())
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'Training rows: {len(X_train)} | Test rows: {len(X_test)}')

## 6. Evaluate the model

In [ ]:
metrics = {
    'MAE_USD': round(float(mean_absolute_error(y_test, y_pred)), 2),
    'RMSE_USD': round(float(np.sqrt(mean_squared_error(y_test, y_pred))), 2),
    'R2': round(float(r2_score(y_test, y_pred)), 4),
}
print(json.dumps(metrics, indent=2))

**Metric interpretation**

- **MAE:** average absolute prediction error in USD; lower is better.
- **RMSE:** prediction error in USD that penalizes large errors more heavily; lower is better.
- **R²:** proportion of target variation explained on held-out data; closer to 1 is generally better, but it can be negative.

Use the actual printed values in the internship report.

## 7. Inspect diagnostic plots

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, y_pred, alpha=0.65)
lo = min(float(y_test.min()), float(np.min(y_pred)))
hi = max(float(y_test.max()), float(np.max(y_pred)))
ax.plot([lo, hi], [lo, hi], '--')
ax.set_title('Actual vs. Predicted House Prices')
ax.set_xlabel('Actual sale price (USD)')
ax.set_ylabel('Predicted sale price (USD)')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'actual_vs_predicted.png', dpi=160)
plt.show()

residuals = np.asarray(y_test) - y_pred
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_pred, residuals, alpha=0.65)
ax.axhline(0, linestyle='--')
ax.set_title('Residuals vs. Predicted Prices')
ax.set_xlabel('Predicted sale price (USD)')
ax.set_ylabel('Residual (actual - predicted, USD)')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'residuals.png', dpi=160)
plt.show()

## 8. Review model coefficients

In [ ]:
regressor = model.named_steps['regressor']
coefficients = pd.DataFrame({'feature': list(X.columns), 'coefficient_USD_per_unit': regressor.coef_})
coefficients.loc[len(coefficients)] = ['intercept', regressor.intercept_]
display(coefficients)
coefficients.to_csv(OUTPUT_DIR / 'model_coefficients.csv', index=False)

A coefficient is the estimated change in predicted price for a one-unit increase in that feature, while holding the other included features constant. It is an association in this fitted model, not proof of causation.

## 9. Save the model and metrics

In [ ]:
metrics.update({
    'model': 'Linear Regression',
    'rows_used': int(len(X)),
    'training_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'features': list(X.columns),
    'target': 'SalePrice',
    'split': '80% training / 20% testing',
    'random_state': 42,
})
joblib.dump({'model': model, 'feature_columns': list(X.columns), 'metrics': metrics}, OUTPUT_DIR / 'house_price_model.joblib')
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved artifacts to:', OUTPUT_DIR.resolve())
print('Files:', [p.name for p in OUTPUT_DIR.iterdir()])

## 10. Run one example prediction

In [ ]:
example_house = pd.DataFrame([[1500, 3, 2.0]], columns=list(X.columns))
example_prediction = float(model.predict(example_house)[0])
print(f'Estimated sale price: ${example_prediction:,.2f} USD')

## 11. Conclusion checklist

Before submission, record your actual MAE, RMSE, and R²; inspect both diagnostic plots; describe at least two limitations; and include a screenshot of the prediction interface if you run `streamlit run app.py`.

**Limitations:** this model uses only three predictors and historical sales from Ames, Iowa. It excludes location, house quality, property age, and many other price factors. It is a learning project, not a current real-estate valuation tool.